# 12 — Blending dos Modelos Tunados
## PRT Seguros

O notebook `11` tunou os hiperparâmetros mas não salvou as probabilidades de validação em disco
(só a métrica final). Aqui retreinamos o **CatBoost tuned** e o **XGBoost tuned** com os mesmos
melhores parâmetros encontrados, desta vez guardando `proba_val` para poder testar blends.
O LightGBM tunado piorou no notebook `11` (AUC caiu), então usamos a versão **original** dele
(`04_lightgbm_proba.ipynb`) no blend.

## 1. Carregar dados `_v2` e retreinar os 2 modelos tunados

In [1]:
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier
import xgboost as xgb

RANDOM_STATE = 42
ID_COL, TARGET_COL = "cod_individuo", "churned"

train = pd.read_csv("dados_processados/train_model_ready_v2.csv")
val = pd.read_csv("dados_processados/val_model_ready_v2.csv")
kaggle = pd.read_csv("dados_processados/kaggle_model_ready_v2.csv")

feature_cols = [c for c in train.columns if c not in (ID_COL, TARGET_COL)]
X_train, y_train = train[feature_cols], train[TARGET_COL]
X_val, y_val = val[feature_cols], val[TARGET_COL]
X_kaggle = kaggle[feature_cols]

X_tr_es, X_es, y_tr_es, y_es = train_test_split(
    X_train, y_train, test_size=0.15, stratify=y_train, random_state=RANDOM_STATE
)
neg_pos_ratio_es = (y_tr_es == 0).sum() / (y_tr_es == 1).sum()

# Melhores parâmetros encontrados no notebook 11 (RandomizedSearchCV)
MELHORES_PARAMS_CAT = {"random_strength": 0.5, "learning_rate": 0.02, "l2_leaf_reg": 9, "depth": 6}
MELHORES_PARAMS_XGB = {"subsample": 0.7, "min_child_weight": 5, "max_depth": 4,
                        "learning_rate": 0.02, "gamma": 0.5, "colsample_bytree": 0.7}


In [2]:
cat_tuned = CatBoostClassifier(
    iterations=3000, **MELHORES_PARAMS_CAT,
    scale_pos_weight=neg_pos_ratio_es, eval_metric="AUC",
    random_seed=RANDOM_STATE, early_stopping_rounds=50, verbose=False,
)
cat_tuned.fit(X_tr_es, y_tr_es, eval_set=(X_es, y_es))

proba_val_cat = cat_tuned.predict_proba(X_val)[:, 1]
proba_kaggle_cat = cat_tuned.predict_proba(X_kaggle)[:, 1]
print(f"CatBoost tuned  -> AUC-ROC val = {roc_auc_score(y_val, proba_val_cat):.4f}")


CatBoost tuned  -> AUC-ROC val = 0.8263


In [3]:
xgb_tuned = xgb.XGBClassifier(
    n_estimators=3000, **MELHORES_PARAMS_XGB,
    scale_pos_weight=neg_pos_ratio_es, tree_method="hist", eval_metric="auc",
    early_stopping_rounds=50, random_state=RANDOM_STATE, n_jobs=-1,
)
xgb_tuned.fit(X_tr_es, y_tr_es, eval_set=[(X_es, y_es)], verbose=False)

proba_val_xgb = xgb_tuned.predict_proba(X_val)[:, 1]
proba_kaggle_xgb = xgb_tuned.predict_proba(X_kaggle)[:, 1]
print(f"XGBoost tuned   -> AUC-ROC val = {roc_auc_score(y_val, proba_val_xgb):.4f}")


XGBoost tuned   -> AUC-ROC val = 0.8253


## 2. Carregar o LightGBM original (não tunado) para completar o blend

Usa `cod_individuo` para casar as linhas — o `val_model_ready_v2.csv` tem as mesmas linhas
(mesmo split) que o `val_model_ready.csv` usado pelo LightGBM original, só que com colunas diferentes.

In [4]:
proba_val_lgb_df = pd.read_csv("dados_processados/proba_val/lightgbm.csv")
proba_val_lgb_df = proba_val_lgb_df.set_index("cod_individuo").reindex(val[ID_COL]).reset_index()
proba_val_lgb = proba_val_lgb_df["proba"].values

print(f"LightGBM original -> AUC-ROC val = {roc_auc_score(y_val, proba_val_lgb):.4f}")


LightGBM original -> AUC-ROC val = 0.8238


## 3. Testar combinações de blend na validação

In [5]:
candidatos = {
    "catboost_tuned": proba_val_cat,
    "xgboost_tuned": proba_val_xgb,
    "lightgbm_original": proba_val_lgb,
}
aucs_individuais = {k: roc_auc_score(y_val, v) for k, v in candidatos.items()}

combinacoes = {
    "cat_tuned_x_xgb_tuned": ["catboost_tuned", "xgboost_tuned"],
    "cat_tuned_x_xgb_tuned_x_lgb": ["catboost_tuned", "xgboost_tuned", "lightgbm_original"],
}

resultados_blend = {}
for nome, membros in combinacoes.items():
    media = np.mean([candidatos[m] for m in membros], axis=0)
    resultados_blend[nome] = roc_auc_score(y_val, media)

# Média ponderada pelo (AUC - 0.5) usando todos os 3
pesos = {m: max(aucs_individuais[m] - 0.5, 0) for m in candidatos}
soma = sum(pesos.values())
pesos = {m: w / soma for m, w in pesos.items()}
media_ponderada = sum(candidatos[m] * pesos[m] for m in candidatos)
resultados_blend["ponderado_3"] = roc_auc_score(y_val, media_ponderada)

print("AUC individuais:")
for m, auc in sorted(aucs_individuais.items(), key=lambda x: -x[1]):
    print(f"  {m:<20} {auc:.4f}")

print("\nAUC dos blends:")
for nome, auc in sorted(resultados_blend.items(), key=lambda x: -x[1]):
    print(f"  {nome:<28} {auc:.4f}")

melhor_individual = max(aucs_individuais, key=aucs_individuais.get)
melhor_blend = max(resultados_blend, key=resultados_blend.get)
print(f"\nMelhor individual: {melhor_individual} ({aucs_individuais[melhor_individual]:.4f})")
print(f"Melhor blend      : {melhor_blend} ({resultados_blend[melhor_blend]:.4f})")


AUC individuais:
  catboost_tuned       0.8263
  xgboost_tuned        0.8253
  lightgbm_original    0.8238

AUC dos blends:
  ponderado_3                  0.8262
  cat_tuned_x_xgb_tuned_x_lgb  0.8262
  cat_tuned_x_xgb_tuned        0.8261

Melhor individual: catboost_tuned (0.8263)
Melhor blend      : ponderado_3 (0.8262)


## 4. Gerar a submissão do melhor blend (usando as probabilidades já calculadas para o Kaggle)

In [6]:
submissao_xgb_tuned = pd.read_csv("submissions/submission_xgboost_tuned.csv").set_index("Id")["probabilidade_churn"]
submissao_cat_tuned = pd.DataFrame({"Id": kaggle[ID_COL], "probabilidade_churn": proba_kaggle_cat}).set_index("Id")["probabilidade_churn"]
submissao_xgb_tuned_arr = pd.DataFrame({"Id": kaggle[ID_COL], "probabilidade_churn": proba_kaggle_xgb}).set_index("Id")["probabilidade_churn"]
submissao_lgb = pd.read_csv("submissions/submission_lightgbm.csv").set_index("Id")["probabilidade_churn"]

kaggle_candidatos = {
    "catboost_tuned": submissao_cat_tuned,
    "xgboost_tuned": submissao_xgb_tuned_arr,
    "lightgbm_original": submissao_lgb,
}

if melhor_blend == "cat_tuned_x_xgb_tuned":
    membros = combinacoes["cat_tuned_x_xgb_tuned"]
    proba_final = pd.concat([kaggle_candidatos[m] for m in membros], axis=1).mean(axis=1)
elif melhor_blend == "cat_tuned_x_xgb_tuned_x_lgb":
    membros = combinacoes["cat_tuned_x_xgb_tuned_x_lgb"]
    proba_final = pd.concat([kaggle_candidatos[m] for m in membros], axis=1).mean(axis=1)
else:  # ponderado_3
    proba_final = sum(kaggle_candidatos[m] * pesos[m] for m in kaggle_candidatos)

submissao_blend_tuned = pd.DataFrame({
    "Id": proba_final.index,
    "probabilidade_churn": proba_final.values,
})
submissao_blend_tuned.to_csv("submissions/submission_blend_tuned.csv", index=False)
print(f"Estratégia usada: {melhor_blend} (AUC-ROC validação = {resultados_blend[melhor_blend]:.4f})")
submissao_blend_tuned.head()


Estratégia usada: ponderado_3 (AUC-ROC validação = 0.8262)


,Id,probabilidade_churn
0,221300000002,0.105676
1,221300000020,0.239135
2,221300000097,0.127500
3,221300000148,0.317063
4,221300000166,0.268504
